# Diffusion Sampler 比較分析

**モデル**: `stabilityai/stable-diffusion-2-1`  
**プロンプト**: `"a photo of an astronaut riding a horse on mars"`  
**リファレンス**: DDIM 250step, seed=42  

## 目次
1. 実験設定の確認
2. 生成画像グリッドの表示
3. PSNR / SSIM ヒートマップ
4. 生成時間の比較
5. NFE vs 品質の散布図
6. DPM-Solver: Singlestep vs Multistep 比較
7. CLIP Score 分析
8. 総合考察

In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from PIL import Image
from IPython.display import display

# 出力ディレクトリ
OUTPUT_DIR = Path("outputs")

# results.json のロード
with open(OUTPUT_DIR / "results.json", encoding="utf-8") as f:
    results = json.load(f)

print(f"エントリ数: {len(results)}")
print("\n最初の 3 エントリ:")
for r in results[:3]:
    print(r)

エントリ数: 31

最初の 3 エントリ:
{'sampler': 'ddim', 'order': None, 'solver_mode': None, 'steps': 250, 'nfe': 250, 'time_sec': 299.722, 'image_path': 'reference.png', 'is_reference': True, 'metrics': {'psnr': inf, 'ssim': 1.0, 'clip_score': 0.7098}}
{'sampler': 'ddpm', 'order': None, 'solver_mode': None, 'steps': 10, 'nfe': 10, 'time_sec': 14.374, 'image_path': 'ddpm_steps10.png', 'is_reference': False, 'metrics': {'psnr': 16.0799, 'ssim': 0.206, 'clip_score': 0.5706, 'kid_overall': None, 'fid_overall': None}}
{'sampler': 'ddpm', 'order': None, 'solver_mode': None, 'steps': 20, 'nfe': 20, 'time_sec': 26.591, 'image_path': 'ddpm_steps20.png', 'is_reference': False, 'metrics': {'psnr': 16.078, 'ssim': 0.2036, 'clip_score': 0.5911, 'kid_overall': None, 'fid_overall': None}}


## 1. 実験設定の確認

In [2]:
import pandas as pd

# results をデータフレームに変換
rows = []
for r in results:
    row = {
        "sampler"   : r.get("sampler"),
        "order"     : r.get("order"),
        "mode"      : r.get("solver_mode"),
        "steps"     : r.get("steps"),
        "nfe"       : r.get("nfe"),
        "time_sec"  : r.get("time_sec"),
        "is_ref"    : r.get("is_reference", False),
    }
    if "metrics" in r:
        row.update(r["metrics"])
    rows.append(row)

df = pd.DataFrame(rows)
df

ModuleNotFoundError: No module named 'pandas'

## 2. 生成画像グリッドの表示

`visualize.py` で生成した `grid.png` を表示します。

In [ ]:
grid_path = OUTPUT_DIR / "grid.png"
if grid_path.exists():
    img = Image.open(str(grid_path))
    plt.figure(figsize=(18, 28))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Sampler Comparison Grid", fontsize=16)
    plt.tight_layout()
    plt.show()
else:
    print("grid.png が見つかりません。先に visualize.py を実行してください。")

## 3. PSNR / SSIM ヒートマップ

リファレンス (DDIM 250step) との pixel-level な類似度を比較します。

**PSNR**: 高いほど参照に近い  
**SSIM**: 高いほど参照に近い (最大 1.0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

SAMPLER_ORDER = [
    "ddpm", "ddim", "euler", "heun", "lms2",
    "dpm_solver_1", "dpm_solver_2_single", "dpm_solver_2_multi",
    "dpm_solver_3_single", "dpm_solver_3_multi"
]
SAMPLER_LABELS = [
    "DDPM", "DDIM", "Euler", "Heun", "LMS-2",
    "DPM-1", "DPM-2S", "DPM-2M", "DPM-3S", "DPM-3M"
]
STEPS = [10, 20, 50]

for ax_idx, metric in enumerate(["psnr", "ssim"]):
    data = np.full((len(SAMPLER_ORDER), len(STEPS)), np.nan)
    for row_i, s in enumerate(SAMPLER_ORDER):
        for col_i, st in enumerate(STEPS):
            for r in results:
                if r.get("sampler") == s and r.get("steps") == st and not r.get("is_reference"):
                    if "metrics" in r and r["metrics"].get(metric) is not None:
                        data[row_i, col_i] = r["metrics"][metric]
    
    ax = axes[ax_idx]
    im = ax.imshow(data, cmap="YlGn" if metric == "psnr" else "Blues", aspect="auto")
    plt.colorbar(im, ax=ax, label=metric.upper())
    ax.set_xticks(range(len(STEPS)))
    ax.set_xticklabels(STEPS)
    ax.set_yticks(range(len(SAMPLER_ORDER)))
    ax.set_yticklabels(SAMPLER_LABELS)
    ax.set_xlabel("Steps")
    ax.set_title(f"{metric.upper()} vs Reference")
    
    for r in range(len(SAMPLER_ORDER)):
        for c in range(len(STEPS)):
            v = data[r, c]
            if not np.isnan(v):
                ax.text(c, r, f"{v:.2f}", ha="center", va="center", fontsize=8)

plt.suptitle("Pixel-Level Quality vs Reference (DDIM 250step)", fontsize=13)
plt.tight_layout()
plt.show()

## 4. 生成時間の比較

各サンプラー × ステップ数の実際の生成時間を比較します。

In [ ]:
x = np.arange(len(SAMPLER_ORDER))
width = 0.25
colors = ["#4e79a7", "#f28e2b", "#e15759"]

fig, ax = plt.subplots(figsize=(14, 5))

for i, steps in enumerate(STEPS):
    times = []
    for s in SAMPLER_ORDER:
        t = 0.0
        for r in results:
            if r.get("sampler") == s and r.get("steps") == steps and not r.get("is_reference"):
                t = r.get("time_sec") or 0.0
        times.append(t)
    ax.bar(x + i * width, times, width, label=f"steps={steps}", color=colors[i], alpha=0.85)

ax.set_xticks(x + width)
ax.set_xticklabels(SAMPLER_LABELS, rotation=20, ha="right")
ax.set_xlabel("Sampler")
ax.set_ylabel("Time (sec)")
ax.set_title("Generation Time per Sampler × Steps")
ax.legend()
plt.tight_layout()
plt.show()

print("\n■ 考察:")
print("  - Heun は NFE=2/step なので同 steps でも 2 倍程度の時間")
print("  - DPM-Solver-3-singlestep は NFE=3/step")
print("  - Multistep サンプラー (LMS-2, DPM-2M, DPM-3M) は NFE=1/step で高効率")

## 5. NFE vs 品質 (PSNR) の散布図

実際の UNet フォワード回数 (NFE) と画質 (PSNR) のトレードオフを可視化します。  
右上 = 高品質・低コスト = 理想的

In [ ]:
cmap = plt.cm.tab10
colors_map = {name: cmap(i / len(SAMPLER_ORDER)) for i, name in enumerate(SAMPLER_ORDER)}
step_sizes = {10: 60, 20: 120, 50: 240}

fig, ax = plt.subplots(figsize=(10, 6))

for s_idx, sampler in enumerate(SAMPLER_ORDER):
    nfe_list, psnr_list = [], []
    for steps in STEPS:
        for r in results:
            if r.get("sampler") == sampler and r.get("steps") == steps and not r.get("is_reference"):
                if "metrics" in r and r["metrics"].get("psnr") is not None:
                    nfe = r.get("nfe", steps)
                    psnr = r["metrics"]["psnr"]
                    nfe_list.append(nfe)
                    psnr_list.append(psnr)
                    ax.scatter(nfe, psnr, c=[colors_map[sampler]], s=step_sizes[steps], alpha=0.85)
    if nfe_list:
        pairs = sorted(zip(nfe_list, psnr_list))
        ax.plot(
            [p[0] for p in pairs], [p[1] for p in pairs],
            color=colors_map[sampler], linewidth=1.2, alpha=0.6,
            label=SAMPLER_LABELS[s_idx]
        )

ax.set_xlabel("NFE (UNet forward passes)")
ax.set_ylabel("PSNR (dB) vs Reference")
ax.set_title("Quality vs NFE — Upper right is better")
ax.legend(loc="lower right", fontsize=8, ncol=2)

# マーカーサイズの凡例を手動追加
for steps, sz in step_sizes.items():
    ax.scatter([], [], c="gray", s=sz, label=f"steps={steps}", alpha=0.5)
ax.legend(loc="lower right", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

## 6. DPM-Solver: Singlestep vs Multistep 比較

Order 2/3 について、singlestep (各ステップで複数回 UNet 呼出) と  
multistep (前ステップ結果を再利用して NFE=1/step) の精度を比較します。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

dpm_pairs = [
    ("dpm_solver_2_single", "dpm_solver_2_multi",  "DPM-Solver-2", 2),
    ("dpm_solver_3_single", "dpm_solver_3_multi",  "DPM-Solver-3", 3),
]

for ax_i, (sk, mk, label, order) in enumerate(dpm_pairs):
    ax = axes[ax_i]
    s_psnr, m_psnr = [], []
    s_nfe, m_nfe = [], []
    for steps in STEPS:
        for r in results:
            if r.get("sampler") == sk and r.get("steps") == steps:
                s_psnr.append(r.get("metrics", {}).get("psnr") or 0)
                s_nfe.append(r.get("nfe", steps))
            if r.get("sampler") == mk and r.get("steps") == steps:
                m_psnr.append(r.get("metrics", {}).get("psnr") or 0)
                m_nfe.append(r.get("nfe", steps))
    
    x = np.arange(len(STEPS))
    w = 0.35
    ax.bar(x - w/2, s_psnr, w, label=f"Singlestep (NFE={order}x)", color="#e15759", alpha=0.85)
    ax.bar(x + w/2, m_psnr, w, label=f"Multistep  (NFE=1x)",      color="#4e79a7", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(STEPS)
    ax.set_xlabel("Steps")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title(f"{label}: Singlestep vs Multistep")
    ax.legend()
    ax.annotate("※ Singlestep は NFE が多いが精度が高い場合あり",
                xy=(0.01, 0.02), xycoords="axes fraction", fontsize=8, color="gray")

plt.suptitle("DPM-Solver Singlestep vs Multistep — PSNR vs Reference", fontsize=12)
plt.tight_layout()
plt.show()

print("\n■ 考察:")
print("  - Multistep は同 steps で NFE が order 倍少ない")
print("  - 少ないステップ数 (steps=10) では multistep の精度低下が目立つ傾向")
print("  - 多ステップ (steps=50) では multistep が singlestep に近い品質を実現")

## 7. CLIP Score 分析

プロンプトと生成画像の意味的整合性を可視化します。  
PSNR と異なり、リファレンスとの pixel 差ではなく**テキスト-画像整合性**を測ります。

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(SAMPLER_ORDER))
width = 0.25
colors = ["#4e79a7", "#f28e2b", "#e15759"]

for i, steps in enumerate(STEPS):
    clip_scores = []
    for s in SAMPLER_ORDER:
        cs = 0.0
        for r in results:
            if r.get("sampler") == s and r.get("steps") == steps and not r.get("is_reference"):
                cs = (r.get("metrics") or {}).get("clip_score") or 0.0
        clip_scores.append(cs)
    ax.bar(x + i * width, clip_scores, width, label=f"steps={steps}", color=colors[i], alpha=0.85)

# リファレンスの CLIP Score を水平線で表示
for r in results:
    if r.get("is_reference") and "metrics" in r:
        ref_clip = r["metrics"].get("clip_score")
        if ref_clip:
            ax.axhline(ref_clip, color="green", linestyle="--", linewidth=1.5, label=f"Reference CLIP={ref_clip:.3f}")

ax.set_xticks(x + width)
ax.set_xticklabels(SAMPLER_LABELS, rotation=20, ha="right")
ax.set_ylabel("CLIP Score")
ax.set_title("CLIP Score (text-image alignment) per Sampler × Steps")
ax.legend()
plt.tight_layout()
plt.show()

## 8. 総合考察

以下のセルではサンプラーごとのまとめを表形式で整理します。

In [ ]:
# steps=20 時点での比較サマリー
TARGET_STEPS = 20

summary_rows = []
for s, label in zip(SAMPLER_ORDER, SAMPLER_LABELS):
    for r in results:
        if r.get("sampler") == s and r.get("steps") == TARGET_STEPS and not r.get("is_reference"):
            m = r.get("metrics", {})
            summary_rows.append({
                "Sampler" : label,
                "NFE"     : r.get("nfe"),
                "Time(s)" : r.get("time_sec"),
                "PSNR"    : m.get("psnr"),
                "SSIM"    : m.get("ssim"),
                "CLIP"    : m.get("clip_score"),
                "KID"     : m.get("kid_overall"),
            })

summary_df = pd.DataFrame(summary_rows).set_index("Sampler")
print(f"■ Steps={TARGET_STEPS} 時の比較サマリー")
display(summary_df.style.highlight_max(subset=["PSNR", "SSIM", "CLIP"], color="lightgreen")
                         .highlight_min(subset=["KID", "Time(s)"], color="lightblue"))

### 結論

| 観点 | 推奨サンプラー |
|------|----------------|
| 最高品質 (多 NFE 許容) | Heun (NFE=2/step) または DPM-Solver-3-singlestep (NFE=3/step) |
| NFE 効率 (少 NFE で高品質) | DPM-Solver-2-multistep / DPM-Solver-3-multistep |
| テキスト整合性 (CLIP Score) | DDIM / DPM-Solver 系 |
| シンプルさ | Euler / DDIM |

**DPM-Solver++ multistep** は steps=20 程度で他の steps=50 サンプラーと同等以上の品質を  
NFE=20 という低コストで実現でき、実用上最もバランスの良い選択肢と言えます。